# Step 2 Diagnostics: SMH
How SMH's IDs link, how the PICANet file links to demographics, what the admission time means (local time against UTC), and whether the medication files are versions of one another. Counts only; nothing here changes any data.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.home() / "Documents/DS217/users/jormond/open_picu/code"))
import diagnostics as D
import manifest as M
import picanet_times as PT
import smh_diagnostics as SD

In [ ]:
base = Path.home() / "Documents/DS217/users/jormond"
open_picu_dir = base / "open_picu"
reports_dir = open_picu_dir / "reports"
smh_dir = base / "SMH"

d = SD.load(smh_dir)
smh_m, smh_w = M.smh_manifest(smh_dir)

## 1. Demographics and the PICANet link
The demographics file repeats rows per encounter (exact duplicates, and some rows that differ, mostly in weight). `encounterId` is the stay key. PICANet starts in 2013, so earlier encounters are outside the cohort. `UniquePtID` is not the same string in the two files, so it cannot be used to link them.

In [ ]:
D.show(SD.demographics_duplicates(d))

In [ ]:
D.show(SD.picanet_link(d))

In [ ]:
D.show(SD.weight_checks(d))

## 2. Times
`inTime` is London local time and `utcChartTime` is UTC. If so, the first vitals row sits one hour before `inTime` in summer and at about zero in winter. The demographics-versus-PICANet time and LOS checks are in `picanet_times.py`.

In [ ]:
D.show(SD.time_offset(smh_dir, d))

In [ ]:
PT.check_times(smh_dir, "SMH", demo_adm="inTime", manifest=smh_m)

## 3. Medication files and drug coverage
`medications_v2` appears to be `medications` plus `medications_furosemide`. Then, for the eligible encounters, the share with at least one row in each drug table, by admission year, to see whether any table thins out.

In [ ]:
SD.medication_versions(smh_dir)

In [ ]:
SD.drug_coverage_by_year(smh_dir, smh_m)

## Exact duplicate rows in the SMH tables
Assembly removes rows that are identical in every column (see `SMH_findings.md`, question 5). Are they copies made by the extraction, or real repeated entries? For the intake and output table (`ai_picu_io`, where amounts are summed and a wrongly removed or kept copy would change totals) this counts how many rows are copies, how many times each row appears, how far apart the copies sit in the file, how many encounters are affected and how the rate changes by admission year. The number of copies should equal the exact duplicates removed in the assembly log (11,822 for this table). Counts only.

In [ ]:
smh_m = pd.read_csv(open_picu_dir / "derived" / "SMH_manifest.csv", dtype=str)
dup_io = SD.duplicate_pattern(smh_dir, smh_m, "ai_picu_io.csv")
D.show(dup_io)

## Exact duplicate rows in SMH vitals
Assembly removes rows that are identical in every column, and for vitals that is 3,441,560 of 8,449,349 rows in manifest stays (40.7%), the second largest share after the ventilation table. This is the table behind the hourly heart rate, blood pressure, temperature and saturation values, so it matters whether those rows are copies made by the extraction or real repeated readings. Same counts as for `ai_picu_io`: how many rows are copies, how many times each row appears, how far apart the copies sit in the file, how many encounters are affected and how the rate changes by admission year. The number of copies should equal the exact duplicates removed in the assembly log (3,441,560). Counts only. Closeness in the file does not by itself separate an extraction artefact from simultaneous real readings, because the time is one of the matching columns; the spread across encounters and years is the better evidence.

In [ ]:
dup_vitals = SD.duplicate_pattern(smh_dir, smh_m, "ai_picu_vitals.csv")
D.show(dup_vitals)

## Do the duplicate vitals rows hold one reading or several?
Assembly removes rows identical in every column. To tell whether those rows are one reading written several times or several real readings, rows of manifest stays are grouped by encounter, time and measurement id (`interventionId`, `attributeId`). Each group is then counted as one row, several identical rows (one reading written more than once), or several rows with different values at the same time (real readings that remain after the exact copies are removed). The time resolution is also counted: the share of times exactly on a whole minute. If nearly all times are whole minutes, a monitor logging more often than once a minute would produce identical rows within one timestamp. Counts only.

In [ ]:
key_vitals = SD.same_key_values(smh_dir, smh_m, "ai_picu_vitals.csv")
D.show(key_vitals)

## SMH ventilation: duplicate rows
Vent has the largest share of exact duplicates (5,293,616 of 9,776,642 rows in manifest stays, 54.1%). Same two checks as for vitals: where the copies sit (`duplicate_pattern`) and whether rows sharing an encounter, time and `interventionId` hold one value or several (`same_key_values`). The number of copies should equal the exact duplicates removed in the assembly log (5,293,616). Counts only.

In [ ]:
dup_vent = SD.duplicate_pattern(smh_dir, smh_m, "ai_picu_vent.csv")
key_vent = SD.same_key_values(smh_dir, smh_m, "ai_picu_vent.csv", id_cols=("interventionId",))
D.show(dup_vent)
D.show(key_vent)

## Are the copies a join repeating every row of an encounter?
`smh_mappings.EXTRACTION` records that every SMH extraction query except `io` is joined to the weight records. A join to a list with k records per encounter would repeat every row of that encounter k times. If so, within an encounter nearly every distinct row appears the same number of times (k), k is the same in vitals and in vent, and k is related to the number of weight records. For each table this counts the commonest repeat count per encounter, how uniform it is within an encounter, and how it compares with the number of demographics rows of the encounter (demographics also repeats rows when an encounter has several weights). Counts only.

In [ ]:
mult_vitals = SD.multiplicity_by_encounter(smh_dir, smh_m, "ai_picu_vitals.csv")
mult_vent = SD.multiplicity_by_encounter(smh_dir, smh_m, "ai_picu_vent.csv")
D.show(mult_vitals)
D.show(mult_vent)

## How much of the duplication does the weight join explain?
The extraction SQL joins each query except `io` to the weight records (`PtDemographic`, `interventionId = 10640`), so every row of an encounter should appear as many times as the encounter has rows in the raw demographics file (n). For vitals and vent this splits the exact copies removed at assembly into the part explained by that join (for each distinct row, the number of times it appears up to n, minus one) and a left-over (identical rows beyond n, for example the same reading written twice in one minute). The two add up to the exact duplicates in the assembly log (3,441,560 for vitals, 5,293,616 for vent). A high share explained, by year as well as overall, confirms that the copies are an artefact of the extraction. Counts only.

In [ ]:
expl_vitals = SD.copies_explained_by_join(smh_dir, smh_m, "ai_picu_vitals.csv")
expl_vent = SD.copies_explained_by_join(smh_dir, smh_m, "ai_picu_vent.csv")
D.show(expl_vitals)
D.show(expl_vent)

## 4. Save

In [ ]:
D.write_report({
    "Demographics": SD.demographics_duplicates(d), "PICANet link": SD.picanet_link(d),
    "Weight and time within an encounter": SD.weight_checks(d), "Time offset": SD.time_offset(smh_dir, d),
    "Medication versions": SD.medication_versions(smh_dir),
    "Drug coverage by year": SD.drug_coverage_by_year(smh_dir, smh_m),
    "Duplicate rows, ai_picu_io": dup_io,
    "Duplicate rows, ai_picu_vitals": dup_vitals,
    "Same-key values, ai_picu_vitals": key_vitals,
    "Duplicate rows, ai_picu_vent": dup_vent,
    "Same-key values, ai_picu_vent": key_vent,
    "Repeat counts, ai_picu_vitals": mult_vitals,
    "Repeat counts, ai_picu_vent": mult_vent,
    "Copies explained by the weight join, ai_picu_vitals": expl_vitals,
    "Copies explained by the weight join, ai_picu_vent": expl_vent,
}, reports_dir / f"SMH_diagnostics_{pd.Timestamp.today().date()}.md", title="SMH diagnostics")